# Lab 8: Text Classification

**Business question:** Can a simple model route a complaint more reliably than a default guess?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | The 24 customer messages from Lab 1 | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | CFPB complaint extract (train and valid splits) and saved LLM labels | about 30–45 min | Three exercises and the interpretation |
| 3 Challenge | CFPB complaints | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

You build a **majority baseline** (always guess the most common label) and a **TF-IDF logistic regression** classifier. Then you compare them with an **embedding classifier** and **saved LLM labels**. You do not call an LLM. I ran the prompt once and saved every answer in `data/llm_outputs`.

You judge every method on the **validation** split. The **test** split stays sealed: you save predictions for it, but you do not score them. Module 9 opens the test split once, at the end.

## Part 1 Guided core

### 1.1 Load the libraries

Select the `ban6403` kernel, then run the cell below. It loads the libraries we use in this lab.

In [ ]:
from pathlib import Path   # file and folder paths
import json                # reads the saved LLM records
import numpy as np         # arrays of numbers
import pandas as pd        # tables
from sklearn.dummy import DummyClassifier                    # the majority baseline
from sklearn.feature_extraction.text import TfidfVectorizer  # the Module 4 word features
from sklearn.linear_model import LogisticRegression          # the classifier
from sklearn.pipeline import make_pipeline                   # chains TF-IDF and the classifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score  # the metrics

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the data is one folder up, in `../data`. Your own results go to `../outputs/m08`.

In [ ]:
# Folder paths, relative to the notebooks folder.
DATA = Path("../data")
OUTPUT_DIR = Path("../outputs/m08")
# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 The target and the partition

A classifier needs a **target**: the label it predicts. Our target is the service team's decision from Modules 2 and 7: does the message need **follow-up** (an open problem or a request for help)? I read each message and listed the 11 that do.

In Module 2 you labeled messages `review`, `no_review` or `unclear`. For this target, `review` and `unclear` become follow-up = yes (an unclear message still needs a reply asking for details), and `no_review` becomes no. My list covers all 23 messages, including the protected ones you did not label, and replaces your own Module 2 labels, so everyone uses the same target.

The model may use only **information available at prediction time**. When a message arrives, the team has its text. So the model sees the text only.

The Module 2 partition says which messages we may learn from (`development`) and which we keep aside to check the model (`protected`). FB022 has no text and is left out.

In [ ]:
# Add each message's Module 2 partition, then keep the 23 messages with text.
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")
partition = pd.read_csv(DATA / "partitions/m02_partition.csv")
feedback = feedback.merge(partition[["feedback_id", "partition"]], on="feedback_id")
feedback = feedback.dropna(subset=["text"]).reset_index(drop=True)

# The target: "yes" for the 11 messages that need follow-up, "no" for the others.
# np.where(condition, a, b) gives a where the condition is True and b elsewhere.
NEEDS_FOLLOW_UP = ["FB001", "FB003", "FB006", "FB008", "FB009", "FB012",
                   "FB014", "FB015", "FB017", "FB018", "FB023"]
feedback["follow_up"] = np.where(feedback["feedback_id"].isin(NEEDS_FOLLOW_UP), "yes", "no")

# Messages by partition and target; margins=True adds the "All" totals.
pd.crosstab(feedback["partition"], feedback["follow_up"], margins=True)

### 1.4 Separate the two partitions

Everything the model learns must come from `dev`. We use `protected` only to check predictions.

The customer messages and the complaints use different names for the same jobs. In Module 2 the protected messages were kept apart for checking. In this lab they play the **validation** role: we check and compare models on them, so they are no longer a fresh test. The customer messages have no test set.

| Job | Course complaints (`split` column) | Customer messages (Module 2 partition) |
|---|---|---|
| Fit vocabulary and weights | `train` (6,000) | `development` (13) |
| Check and compare models (validation) | `valid` (2,000) | `protected` (10) |
| Final check, once (test) | `test` (2,000), sealed until Module 9 | none |
| Fresh check of a frozen workflow | Module 13 held-out set: 60 `test` complaints with new labels | none |

In [ ]:
# .copy() makes separate tables, so we can add prediction columns to them later.
dev = feedback[feedback["partition"].eq("development")].copy()
protected = feedback[feedback["partition"].eq("protected")].copy()
print("development:", len(dev), "| protected:", len(protected))

### 1.5 The majority baseline

A **baseline** is the simplest reasonable method. Any real model has to beat it. The **majority baseline** always predicts the label that is most common in the development data. scikit-learn calls it `DummyClassifier`. It ignores the text completely.

**Accuracy** is the share of predictions that are correct. **Macro F1** averages one score per class, so a model that never predicts "yes" gets a low score even when accuracy looks fine. You meet both again in step 1.8.

In [ ]:
# strategy="most_frequent": always predict the most common label in the data it is fitted on.
baseline = DummyClassifier(strategy="most_frequent")
# fit(X, y): X is the input (here a one-column table of text, which the baseline ignores), y the labels.
baseline.fit(dev[["text"]], dev["follow_up"])
# predict gives one label per protected message.
protected["baseline"] = baseline.predict(protected[["text"]])
print("baseline always predicts:", protected["baseline"].unique().tolist())

# Each metric takes the true labels first, then the predictions. average="macro" gives macro F1.
print("accuracy:", accuracy_score(protected["follow_up"], protected["baseline"]),
      "| macro F1:", round(f1_score(protected["follow_up"], protected["baseline"], average="macro"), 2))

### 1.6 TF-IDF and logistic regression in one Pipeline

A **Pipeline** chains steps so they always run in the same order. Ours has two steps:

1. `TfidfVectorizer` turns each text into TF-IDF weights (Module 4). It learns its vocabulary in `fit`.
2. `LogisticRegression` learns one weight per word. A positive weight pushes a message toward "yes", a negative weight toward "no".

`pipe.fit(...)` fits both steps on the development messages only. Later, `pipe.predict(...)` reuses that vocabulary and those weights. Words that only appear in protected messages are simply ignored.

In [ ]:
# The two steps, in order: TF-IDF, then logistic regression.
pipe = make_pipeline(TfidfVectorizer(), LogisticRegression())
# fit runs fit_transform for TF-IDF, then fits the classifier, on the development messages only.
pipe.fit(dev["text"], dev["follow_up"])

# named_steps gives each step by its name; vocabulary_ holds the words TF-IDF learned.
vocab = pipe.named_steps["tfidfvectorizer"].vocabulary_
print("words learned from development messages:", len(vocab))
print("is 'package' (only in FB014) in the vocabulary?", "package" in vocab)

### 1.7 Predict the protected messages

`predict` gives a label. `predict_proba` gives a probability for each class, in the order of `pipe.classes_`. Column 1 is "yes". We keep `p_yes` at full precision and round it only for display.

In [ ]:
print("class order:", pipe.classes_)
protected["predicted"] = pipe.predict(protected["text"])

# predict_proba gives one column per class; [:, 1] takes column 1 ("yes") for every row.
protected["p_yes"] = pipe.predict_proba(protected["text"])[:, 1]

# Round p_yes to 2 decimals for display only.
protected[["feedback_id", "follow_up", "predicted", "p_yes", "text"]].round({"p_yes": 2})

### 1.8 Confusion matrix, per-class recall and macro F1

- A **confusion matrix** counts each pair of (true label, predicted label). Rows are the true labels, columns the predictions. The diagonal holds the correct predictions.
- **Recall** for a class is the share of that class the model found: correct "yes" predictions divided by all true "yes" messages.
- **F1** for a class combines recall with **precision** (the share of "yes" predictions that are right): F1 = 2 × precision × recall / (precision + recall). It is high only when both are high. **Macro F1** is the plain average of the F1 of each class, so small classes count as much as large ones.

Check one class by hand after you run the cell. For "yes", the pipeline makes 4 "yes" predictions and 2 are right, so precision = 2/4 = 0.5. It finds 2 of the 5 true "yes" messages, so recall = 2/5 = 0.4. F1 = 2 × 0.5 × 0.4 / (0.5 + 0.4) = 0.44.

In [ ]:
# labels fixes the order of the rows and columns: "no" first, then "yes".
labels = ["no", "yes"]
cm = confusion_matrix(protected["follow_up"], protected["predicted"], labels=labels)
print(pd.DataFrame(cm, index=["true no", "true yes"], columns=["predicted no", "predicted yes"]))

# average=None gives one recall per class, in the order of labels. :.2f shows 2 decimals.
recall = recall_score(protected["follow_up"], protected["predicted"], labels=labels, average=None)
print(f"recall: no {recall[0]:.2f} | yes {recall[1]:.2f}")
print("macro F1: pipeline", round(f1_score(protected["follow_up"], protected["predicted"], average="macro"), 2),
      "| baseline", round(f1_score(protected["follow_up"], protected["baseline"], average="macro"), 2))

### 1.9 Which words did the model learn?

Logistic regression is easy to inspect: each word has one weight. Look at the five most negative weights (toward "no") and the five most positive (toward "yes"). Do they make business sense, or did the model learn accidents of 13 short messages?

In [ ]:
# The words, in the same order as the classifier's weights.
words = pipe.named_steps["tfidfvectorizer"].get_feature_names_out()
# coef_[0] holds one weight per word; sort from the most negative to the most positive.
weights = pd.Series(pipe.named_steps["logisticregression"].coef_[0], index=words).sort_values()

# Write each word with its weight, for example "please +0.28" (:+.2f shows the sign and 2 decimals).
toward_no = []
for word, weight in weights.head(5).items():
    toward_no.append(f"{word} {weight:+.2f}")
toward_yes = []
for word, weight in weights.tail(5).items():
    toward_yes.append(f"{word} {weight:+.2f}")
print("toward no: ", ", ".join(toward_no))
print("toward yes:", ", ".join(toward_yes))

### 1.10 Read the mistakes

Show the protected messages where the prediction is wrong. For each one, ask: which words could the model have used, and were they in the development messages?

In [ ]:
# Rows where the prediction differs from the true label.
is_wrong = protected["predicted"].ne(protected["follow_up"])
protected.loc[is_wrong, ["feedback_id", "follow_up", "predicted", "p_yes", "text"]].round({"p_yes": 2})

### 1.11 Save your outputs

This writes the protected predictions to `outputs/m08`, with `p_yes` at full precision. The data files are not changed. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
save_cols = ["feedback_id", "follow_up", "baseline", "predicted", "p_yes", "text"]
protected[save_cols].to_csv(OUTPUT_DIR / "feedback_predictions.csv", index=False)
print("saved:", OUTPUT_DIR / "feedback_predictions.csv")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. Compare the baseline and the pipeline on the protected messages. Which number shows the difference better, accuracy or macro F1, and why?
2. The protected set has only 10 messages. What does one changed prediction do to the recall for "yes"? What does that tell you about these results?

**Your answer:**

## Part 2 Practice: routing real complaints

Lab 7 practised on clothing reviews. This lab returns to the CFPB complaints in `data/course/cfpb_complaints.csv`.

Now the real task. A complaint team receives each complaint and must send it to the right **product team**. The target is the `product` field (six products). We call it the **route**.

Two cautions before you start:
- The consumer chose the product from a menu. It is usually right, but not always. The label itself can be wrong. This is **label noise**.
- The model may only use information that exists when the complaint arrives. Step 2.2 checks which columns qualify.

### 2.1 Load the complaints and the splits

We give each product a short route name. `train` is for fitting and `valid` is for checking. `test` stays sealed.

In [ ]:
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")

# A short route name for each product.
SHORT = {"Checking or savings account": "checking", "Credit card": "credit_card",
         "Credit reporting or other personal consumer reports": "credit_reporting",
         "Debt collection": "debt_collection",
         "Money transfer, virtual currency, or money service": "money_transfer", "Mortgage": "mortgage"}
complaints["route"] = complaints["product"].map(SHORT)
# The six routes in alphabetical order; we use this order in every table.
ROUTES = sorted(SHORT.values())

# train and valid keep the row numbers of `complaints` (their index).
train = complaints[complaints["split"].eq("train")].copy()
valid = complaints[complaints["split"].eq("valid")].copy()

# Complaints by route and split; margins=True adds the totals.
pd.crosstab(complaints["route"], complaints["split"], margins=True)

### 2.2 Leakage audit: which columns can the model use?

**Leakage** means the model uses information it would not have at prediction time, or information that already contains the answer. A leaky model looks excellent in the lab and fails in use.

| Column | Known when the complaint arrives? | Use as a feature? |
|---|---|---|
| `narrative` | Yes, the consumer's text | **Yes** |
| `issue`, `sub_issue`, `sub_product` | Chosen in the same form, from a menu that depends on the product | No: they contain the answer |
| `company_response`, `timely_response` | No, only after the company replies | No: they come later |
| `company`, `state`, `submitted_via` | Yes | Not in this lab (text only) |

The cell shows how strong the `issue` leak is. It guesses the route from the most common product for each issue in `train`.

In [ ]:
def most_common(routes):
    """Take a column of routes; return the most common one."""
    # value_counts sorts from the most common down; index[0] is the first route.
    return routes.value_counts().index[0]

# For each issue, the most common route among the training complaints.
issue_lookup = train.groupby("issue")["route"].agg(most_common)
# Guess each validation complaint's route from its issue.
issue_guess = valid["issue"].map(issue_lookup)

routes_per_issue = complaints.groupby("issue")["route"].nunique()
print("issues:", complaints["issue"].nunique(), "| issues used by more than one product:",
      (routes_per_issue > 1).sum())
# The mean of True/False values is the share of correct guesses.
print("valid accuracy of the issue lookup:", round((issue_guess == valid["route"]).mean(), 3))

### 2.3 The label is the consumer's choice

The same problem can be filed under different products. The cell counts complaints whose issue is about a **credit report**, by the product the consumer chose.

In [ ]:
report_issues = ["Incorrect information on your report", "Improper use of your report",
                 "Problem with a company's investigation into an existing problem",
                 "Problem with a company's investigation into an existing issue"]
# Complaints with a credit-report issue, counted by the product the consumer chose.
is_report_issue = complaints["issue"].isin(report_issues)
complaints.loc[is_report_issue, "route"].value_counts()

### 2.4 Majority baseline on the complaints

Fit the majority baseline on `train` and check it on `valid`.

In [ ]:
# The same baseline as 1.5, fitted on train.
route_baseline = DummyClassifier(strategy="most_frequent")
route_baseline.fit(train[["narrative"]], train["route"])
valid["baseline_route"] = route_baseline.predict(valid[["narrative"]])
print("baseline always predicts:", valid["baseline_route"].unique().tolist())
print("valid accuracy:", accuracy_score(valid["route"], valid["baseline_route"]),
      "| macro F1:", round(f1_score(valid["route"], valid["baseline_route"], average="macro"), 3))

### 2.5 TF-IDF logistic regression on the complaints

The TF-IDF settings for this lab are: words and two-word phrases (`ngram_range=(1, 2)`), a term must appear in at least 5 training complaints (`min_df=5`), and the two most common redaction marks are left out (`stop_words=["xx", "xxxx"]`). CFPB writes `XXXX` where it hid a name or a number, so these marks say nothing about the product. Common words such as *not* stay. `max_iter=1000` gives the fitting enough steps to finish. Fitting takes a few seconds.

In [ ]:
tfidf_pipe = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),         # words and two-word phrases
        min_df=5,                   # keep terms that appear in at least 5 training complaints
        stop_words=["xx", "xxxx"],  # drop the redaction marks
    ),
    LogisticRegression(max_iter=1000),  # max_iter: enough steps for the fit to finish
)
# Fit on train only, then predict valid.
tfidf_pipe.fit(train["narrative"], train["route"])
valid["tfidf_route"] = tfidf_pipe.predict(valid["narrative"])
print("vocabulary size:", len(tfidf_pipe.named_steps["tfidfvectorizer"].vocabulary_))
print("valid accuracy:", round(accuracy_score(valid["route"], valid["tfidf_route"]), 3),
      "| macro F1:", round(f1_score(valid["route"], valid["tfidf_route"], average="macro"), 3))

### 2.6 Where does the model go wrong?

Rows are the true routes, columns the predicted routes. Read the largest numbers off the diagonal. Then read the recall per route.

In [ ]:
# Rows = true routes, columns = predicted routes, both in ROUTES order.
cm = confusion_matrix(valid["route"], valid["tfidf_route"], labels=ROUTES)
print(pd.DataFrame(cm, index=ROUTES, columns=ROUTES).to_string())
print()

# One recall per route, in ROUTES order.
recalls = recall_score(valid["route"], valid["tfidf_route"], labels=ROUTES, average=None)
pd.Series(recalls, index=ROUTES, name="recall").round(3)

### 2.7 Words that push toward each route

Each route has its own row of weights. The cell lists the six words or phrases with the largest weights for each route.

In [ ]:
features = tfidf_pipe.named_steps["tfidfvectorizer"].get_feature_names_out()
# coef_ has one row of weights per route, in the order of tfidf_pipe.classes_.
coefs = tfidf_pipe.named_steps["logisticregression"].coef_
# zip pairs each route with its row of weights, in order.
for route, route_weights in zip(tfidf_pipe.classes_, coefs):
    # argsort returns positions from smallest weight to largest; reversing ([::-1]) and taking six ([:6])
    # selects the positions of the six largest weights.
    top_positions = np.argsort(route_weights)[::-1][:6]
    # features[top_positions] retrieves the words at those positions.
    top_words = ", ".join(features[top_positions])
    # {route:>17} right-aligns the route name in 17 characters.
    print(f"{route:>17}: {top_words}")

### 2.8 Save validation and test probabilities for Module 9

`predict_proba` gives one probability per route. We save them for `valid` and for `test`, at full precision, because Module 9 makes decisions from them. We do **not** score the test predictions. Module 9 opens them once, after its policy is fixed. The repository also holds a frozen copy of these predictions in `data/predictions` (rounded to 4 decimals), so Module 9 does not depend on your run.

In [ ]:
def probability_table(rows):
    """Take some complaints; return their IDs, true route, predicted route and one probability per route."""
    # One column per route, named p_checking, p_credit_card, ...
    columns = []
    for route in tfidf_pipe.classes_:
        columns.append("p_" + route)
    proba = pd.DataFrame(tfidf_pipe.predict_proba(rows["narrative"]), columns=columns)

    table = rows[["complaint_id", "split", "route"]].reset_index(drop=True)
    table["tfidf_route"] = tfidf_pipe.predict(rows["narrative"])
    # axis=1 puts the two tables side by side.
    return pd.concat([table, proba], axis=1)

# Save valid and test predictions; the test predictions are not scored here.
test = complaints[complaints["split"].eq("test")]
probability_table(valid).to_csv(OUTPUT_DIR / "route_tfidf_valid.csv", index=False)
probability_table(test).to_csv(OUTPUT_DIR / "route_tfidf_test.csv", index=False)
print("saved:", OUTPUT_DIR / "route_tfidf_valid.csv")
print("saved:", OUTPUT_DIR / "route_tfidf_test.csv")

### 2.9 Load the cached LLM labels

I sent 300 validation complaints to gemini-3.5-flash-lite with the prompt `m08_route_v1` (in `data/llm_outputs/prompts/`). The 300 were drawn so that each product keeps its share of the validation split. The draw also kept the share of urgent complaints (fraud, scams, identity theft) that Module 9 uses. The prompt gives a short definition of each product and asks for exactly one label. It sends only the narrative. This is **zero-shot** classification: the LLM gets no labeled examples.

Each record stores the model, the prompt version, the date and the time taken (`data/llm_outputs/README.md` describes every field). You will compare every method on these same 300 validation complaints.

In [ ]:
# Read the 300 saved records, one JSON record per line (as in Lab 7).
llm_records = []
for line in (DATA / "llm_outputs/m08_route_valid_gemini.jsonl").read_text().splitlines():
    llm_records.append(json.loads(line))

# For each record: count errors, and keep the ID and the LLM's route.
n_errors = 0
complaint_ids = []
llm_routes = []
for rec in llm_records:
    if rec["error"] is not None:
        n_errors += 1
    complaint_ids.append(int(rec["id"]))
    llm_routes.append(SHORT[rec["parsed"]["product"]])

first = llm_records[0]
print("records:", len(llm_records), "| errors:", n_errors)
print("model:", first["model_version"], "| prompt:", first["prompt_version"], "| temperature:", first["temperature"])

# The 300 validation complaints with their LLM route.
llm = pd.DataFrame({"complaint_id": complaint_ids, "llm_route": llm_routes})
subset = valid.merge(llm, on="complaint_id")
print("complaints in the LLM subset:", len(subset))
subset["route"].value_counts()

### Exercise 1: an embedding classifier

Train a second classifier on the saved sentence embeddings of the complaints (`data/embeddings/cfpb_minilm.npy`, one row per complaint in the same order as the CSV). Load them with `.astype("float32")`, which stores each vector value as a 32-bit decimal number. They were made with the same model as the review vectors in Lab 5. You trained a similar classifier on weak labels from star ratings in Lab 7 (step 2.7). This time the labels are the routes.

1. Fit `LogisticRegression(max_iter=1000)` on the `train` rows. Hint: `cfpb_vectors[train.index]` selects them, because `train` keeps the row numbers of `complaints`.
2. Predict the `valid` rows and store the result in `valid["emb_route"]`.
3. Print the valid accuracy and macro F1, and the recall per route, as in 2.5 and 2.6.

In [ ]:
# Exercise 1: your code here
# TODO: load the vectors: cfpb_vectors = np.load(DATA / "embeddings/cfpb_minilm.npy").astype("float32")
# TODO: fit LogisticRegression(max_iter=1000) on cfpb_vectors[train.index] and train["route"].
# TODO: predict cfpb_vectors[valid.index] and store the result in valid["emb_route"].
# TODO: print the valid accuracy and macro F1, and show the recall per route (as in 2.5 and 2.6).

### Exercise 2: compare all four methods on the same complaints

The LLM labels exist only for the 300 complaints in `subset`. A fair comparison uses the **same** complaints for every method.

1. Add the embedding predictions to `subset` (merge on `complaint_id`) and add a `majority` column with the baseline's guess.
2. Make one table with a row per method (majority, TF-IDF, embedding, LLM) and these columns: macro F1 on the subset, and the recall for each route.
3. Below the table, print the macro F1 of the three non-LLM methods on the **full** validation split (2,000 complaints) as well.

Hint: `recall_score(subset["route"], subset[column], labels=ROUTES, average=None)` gives the six recalls in `ROUTES` order.

A partial example for one method. Each method gives one row: a dictionary with the method name, the macro F1 and one recall per route. `zip` pairs the items of two lists in order, here each route with its recall.

```python
recalls = recall_score(subset["route"], subset["tfidf_route"], labels=ROUTES, average=None)
# Start one results row with the method name and overall score.
row = {"method": "TF-IDF",
       "macro_f1": f1_score(subset["route"], subset["tfidf_route"], average="macro")}
# Add one recall value per route.
for route, recall in zip(ROUTES, recalls):
    row[route] = recall
```

Make one such row for each of the four methods, collect the rows in a list, and turn the list into a table with `pd.DataFrame(rows)`.

In [ ]:
# Exercise 2: your code here
# TODO: merge valid[["complaint_id", "emb_route"]] into subset, and add a "majority" column
#       with route_baseline.predict(subset[["narrative"]]).
# TODO: for each method column (majority, tfidf_route, emb_route, llm_route), build one row
#       as in the example above (method, macro_f1, one recall per route); collect the rows in a list.
# TODO: turn the list into one table with pd.DataFrame(rows), one row per method.
# TODO: print the macro F1 of majority (baseline_route), TF-IDF and embedding on all of valid.

### Exercise 3: inspect five consequential errors

Not every error costs the same. A complaint about stolen money that lands on the mortgage team waits longer than one that lands on a related team.

1. Find complaints in `subset` where the TF-IDF route is wrong. Sort them so the most **confident** wrong predictions come first. Hint: `tfidf_pipe.predict_proba(...)` and `.max(axis=1)`, which takes the largest class probability in each complaint's row. This probability ranks the model's confidence. It does not show that the prediction is correct, or that the business cost is high.
2. Pick five errors that would matter to the business. Show `complaint_id`, `route`, `tfidf_route`, `emb_route`, `llm_route` and the first 400 characters of the narrative.
3. In a markdown cell or a comment, write one line per complaint. Is it a **model error**, where the text clearly says the true route? Or is it a **label question**, where the text fits the predicted route as well or better? Then say what the misroute would cost.

In [ ]:
# Exercise 3: your code here
# TODO: add the TF-IDF confidence to subset: tfidf_pipe.predict_proba(subset["narrative"]).max(axis=1).
# TODO: keep the rows where tfidf_route differs from route, sorted by confidence, highest first.
# TODO: pick five errors and show complaint_id, route, tfidf_route, emb_route, llm_route
#       and the first 400 characters of the narrative (.str[:400]).
# TODO: write one line per complaint: model error or label question, and the cost of the misroute.

### Practice interpretation

Write 3–5 sentences:

- Using 2.2 and 2.3, which columns did you exclude, and why is the `product` label itself uncertain? Give a number.
- Using 2.4–2.6 and Exercise 2, how much better than the majority baseline is each method in macro F1, and which route is hardest? Which method would you use, and why?
- Using Exercise 3, how many of your five errors are label questions rather than model errors? What does that mean for the macro F1 numbers?

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Naive Bayes in the same Pipeline

Multinomial Naive Bayes is a fast, classic text classifier. Swap it in for logistic regression and compare the valid macro F1.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Same TF-IDF settings as 2.5; only the classifier changes.
nb_pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=5, stop_words=["xx", "xxxx"]), MultinomialNB())
nb_pipe.fit(train["narrative"], train["route"])
nb_pred = nb_pipe.predict(valid["narrative"])
print("Naive Bayes valid macro F1:", round(f1_score(valid["route"], nb_pred, average="macro"), 3))

### B. A regularization path

`C` controls how strongly logistic regression shrinks its weights: a small `C` means strong shrinking (simpler model), a large `C` means weak shrinking. The default is 1. Refit with four values and compare the valid macro F1. This takes about half a minute. We reuse one TF-IDF matrix to save time; that is fine here because the vectorizer is fitted on `train` only.

In [ ]:
# Fit TF-IDF once on train (fit_transform), and only transform valid.
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=5, stop_words=["xx", "xxxx"])
X_train = vectorizer.fit_transform(train["narrative"])
X_valid = vectorizer.transform(valid["narrative"])

for C in [0.1, 1, 10, 100]:
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train, train["route"])
    macro_f1 = f1_score(valid["route"], model.predict(X_valid), average="macro")
    # {C:>5} right-aligns C in 5 characters; :.3f shows 3 decimals.
    print(f"C = {C:>5}: valid macro F1 {macro_f1:.3f}")

### C. Fine-tune a small transformer

Fine-tuning trains all the weights of a small language model, such as DistilBERT, on your labels. It usually needs a GPU and the `transformers` and `torch` packages, which are not in the course environment. The Hugging Face text classification guide in Further reading shows the steps. If you try it, fit on `train`, choose settings on `valid`, and leave `test` sealed.

## Further reading

- [scikit-learn user guide: Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- [scikit-learn user guide: classification metrics (confusion matrix, precision, recall, F1)](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
- [Jurafsky and Martin, Speech and Language Processing, chapters on Naive Bayes and logistic regression](https://web.stanford.edu/~jurafsky/slp3/)
- [Hugging Face guide: text classification](https://huggingface.co/docs/transformers/tasks/sequence_classification)

Next, Module 9 turns predicted probabilities into a decision for an urgent-complaint queue and opens the sealed test split once.